<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/09a_vae_selfies.ipynb)

# 09a · 分子的连续潜空间：SELFIES 表示与变分自编码器（VAE）

在第 08 讲（`08a` ~ `08c`）中，我们构建了直接工作在离散 token 序列上的自回归语言模型与编码器-解码器 Transformer。无论是 `SmilesRNN` 还是 `ReactionTransformer`，它们在生成分子时都是逐个字符推进的——网络内部并不存在一个能够概括整个分子全局结构的固定维度向量表示 $\mathbf{z} \in \mathbb{R}^d$。

然而在材料逆向设计（Inverse Design）中，研究者常常希望把离散的分子或晶体映射到一个**连续光滑的欧氏潜空间（Continuous Latent Space）**中：
- 如果每个分子都对应潜空间中的一个坐标点 $\mathbf{z}$，我们就可以在两个已知材料 $\mathbf{z}_A$ 与 $\mathbf{z}_B$ 之间连续插值，探索折中构型；
- 更进一步，如果能在潜空间上建立“潜坐标 $\mathbf{z} \to$ 材料性质 $y$”的可微映射，我们就能直接沿着目标性质的梯度方向搜索新材料，再由解码器还原为具体的化学结构。

实现这一构想的经典生成模型是**变分自编码器（Variational Autoencoder, VAE；Kingma & Welling, 2014；Gómez-Bombarelli et al., *ACS Central Science*, 2018）**。本节我们将沿着以下脉络展开：
1. **从 SMILES 到 SELFIES**：为什么一次性并行解码的潜变量模型极易触发 SMILES 语法崩溃，以及 **SELFIES**（Krenn et al., 2020）如何从形式语法层面保证 100% 的化学价态合法性；
2. **从确定性自编码器到变分自编码器**：重参数化技巧（Reparameterization Trick）、证据下界（ELBO）与 $\beta$-VAE 的重构—规整权衡；
3. **训练分子 VAE 与重构误差剖析**：检验模型对共轭骨架的记忆能力（内嵌 **练习 09a-1**）；
4. **潜空间高维几何**：为什么欧氏直线插值会在高维球壳内部发生“范数塌陷”，以及如何用球面线性插值（Slerp）修正（内嵌 **练习 09a-2**）；
5. **先验空洞（Prior Hole）与生成指标盲区**：为什么一个 `Valid / Unique / Novel` 三项指标接近满分的生成模型，采样出的分子在化学上可能毫无意义（内嵌 **练习 09a-3**）。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "09 - Generative Models"))
    get_ipython().run_line_magic("pip", "install -q rdkit selfies")

In [ ]:
try:
    import selfies as sf
except ImportError:
    %pip install -q selfies rdkit
    import selfies as sf

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
print("selfies version:", sf.__version__)

<!-- ai4m-style -->
> **课程统一绘图风格**：下方单元加载 `ai4m.mplstyle`（无上右边框、柔和网格与统一科研配色 `C`）。

In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))
import ai4m_style as ai4m

C = ai4m.use()

In [ ]:
DATA_DIR = "data" if os.path.exists("data/cep.csv") else "course/09 - Generative Models/data"
df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print(f"Loaded CEP dataset: {len(df):,} organic photovoltaic molecules")
df.head(3)

---
## 1. 为什么在 VAE 中使用 SELFIES：从脆弱语法到 100% 合法的形式语法

在 `08a` 中，训练充分的 LSTM 语言模型每一步都以前面已生成的全部历史为条件逐字符预测，即便如此仍有约 3%~8% 的生成序列因为括号未闭合、环编号落单或凯库勒化失败而被 RDKit 判定为无效。而在变分自编码器（VAE）中，如果我们用一个前馈解码器从连续潜向量 $\mathbf{z}$ 一次性并行输出所有位置的 token 概率，位置之间缺乏自回归协调，直接输出 SMILES 时的语法崩溃率往往高达 50% 以上。

为了从表示源头消除语法错误，Krenn 等人（*Machine Learning: Science and Technology*, 2020）提出了 **SELFIES（SELF-referencIng Embedded Strings）**。它是一套基于有穷状态自动机的形式化学语法，核心设计包含三点：

1. **显式键级与价态上限跟踪**：每个原子符号明确携带成键方式（如 `[C]`、`[=C]`、`[#N]`），解码器在读取序列时实时维护当前原子的剩余可用化合价；
2. **无配对符号的相对长度索引**：支链和成环不再依赖必须成对出现的括号 `()` 或环号 `1..9`，而是改用指令符 `[Branch1]`、`[Ring1]`，后面紧跟一个代表“支链包含几个原子”或“向前回溯几个原子成环”的长度数值 token；
3. **价态溢出自动截断/降级**：当后续指令试图建立超过当前原子剩余化合价的化学键时，状态机自动将双键降为单键或忽略多余分支，从而在数学上保证**任意 SELFIES 符号序列都能唯一解码为一个符合化合价规则的合法分子图**。

我们先看几个有机半导体分子在 SMILES 与 SELFIES 之间的双向无损转换，随后通过一个随机单点突变实验直观对比两者的语法鲁棒性。

In [ ]:
examples = ["c1ccsc1", "c1ccc2c(c1)sc1c3ccccc3sc21", df["smiles"].iloc[0]]
for s in examples:
    encoded = sf.encoder(s)
    decoded = sf.decoder(encoded)
    print(f"Original SMILES : {s}")
    print(f"SELFIES         : {encoded}")
    print(f"Decoded SMILES  : {decoded}\n")

In [ ]:
# 随机突变实验：将一个合法有机半导体分子（BTBT）中的随机 1 个 token 替换为随机符号
random.seed(0)
btbt_smiles = "c1ccc2c(c1)sc1c3ccccc3sc21"
btbt_selfies = sf.encoder(btbt_smiles)
sf_tokens = list(sf.split_selfies(btbt_selfies))
alphabet_demo = sorted(sf.get_semantic_robust_alphabet())

n_trials = 500
smiles_ok = 0
selfies_ok = 0

for _ in range(n_trials):
    # 1. 随机修改 SMILES 的一个字符
    pos = random.randrange(len(btbt_smiles))
    mut_smi = btbt_smiles[:pos] + random.choice(list("CcNnos12()=")) + btbt_smiles[pos + 1 :]
    if Chem.MolFromSmiles(mut_smi) is not None:
        smiles_ok += 1

    # 2. 随机修改 SELFIES 的一个 token
    toks = sf_tokens.copy()
    toks[random.randrange(len(toks))] = random.choice(alphabet_demo)
    dec_smi = sf.decoder("".join(toks))
    if dec_smi and Chem.MolFromSmiles(dec_smi) is not None:
        selfies_ok += 1

print(f"Validity after 1 random token mutation ({n_trials} trials):")
print(f"  SMILES  : {smiles_ok / n_trials:.1%}")
print(f"  SELFIES : {selfies_ok / n_trials:.1%}")

实验结果非常直观：对 SMILES 随机改动一个字符后，仅剩不到 20% 还能被 RDKit 解析；而对 SELFIES 随机替换任意 token，解码后的分子 **100.0% 依然是化合价合法的分子**。

不过需要提前树立一个贯穿本节的核心警惕：**语法与化合价“合法（Valid）”绝不等于材料化学意义上的“合理（Realistic）”**。我们在第 5 节将会看到，过分迷信 100% 的有效率指标会掩盖多么严重的分布失真。

下面我们将整个 CEP 数据集转换为 SELFIES token 序列，并建立词表（以 `[nop]` 作为长度填充符，索引为 `0`）。

In [ ]:
selfies_list = [sf.encoder(s) for s in df["smiles"]]
token_lists = [list(sf.split_selfies(s)) for s in selfies_list]

alphabet = ["[nop]"] + sorted({t for ts in token_lists for t in ts})
stoi = {t: i for i, t in enumerate(alphabet)}
max_len = max(len(ts) for ts in token_lists)

print(f"SELFIES alphabet size : {len(alphabet)}")
print(f"Max sequence length   : {max_len} (mean = {np.mean([len(ts) for ts in token_lists]):.1f})")
print(f"Alphabet tokens       : {alphabet}")

---
## 2. 从确定性自编码器到变分自编码器（VAE）

### 2.1 为什么普通自编码器不能直接用来生成新分子？

一个**确定性自编码器（Deterministic Autoencoder）**由编码器 $E_\phi(\mathbf{x})$ 与解码器 $D_\theta(\mathbf{z})$ 串联而成：编码器将高维离散分子表示 $\mathbf{x}$ 压缩为低维瓶颈向量 $\mathbf{z} \in \mathbb{R}^d$，解码器再将其还原为 $\hat{\mathbf{x}}$。这相当于我们在 `03a` 学过的主成分分析（PCA）的非线性神经网络推广。

然而，只最小化重构误差的自编码器在潜空间中会把训练样本映射成一座座孤立的“小岛”——编码器为了方便解码器区分不同分子，倾向于把各个样本推得很远。当我们在训练完成后随机从潜空间的空白地带抽样一个 $\mathbf{z}$，或者在两座孤岛之间取中点时，解码器从未在这些空白区域受过训练，解码结果往往是结构破碎的无意义产物。

### 2.2 变分自编码器（VAE）的两大核心机制

为了让潜空间变得连续、紧凑且可采样，**变分自编码器（VAE）**引入了两项改造：

1. **概率化编码与局部邻域平滑**：编码器对每个输入分子 $\mathbf{x}$ 不再输出单个确定性的点，而是输出一个对角高斯后验分布 $q_\phi(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}\big(\boldsymbol{\mu}(\mathbf{x}),\, \text{diag}(\boldsymbol{\sigma}^2(\mathbf{x}))\big)$ 的均值向量 $\boldsymbol{\mu}$ 与对数方差 $\log \boldsymbol{\sigma}^2$。训练时从该分布中随机采样 $\mathbf{z}$ 送入解码器，迫使解码器把 $\boldsymbol{\mu}(\mathbf{x})$ 周围整个邻域内的点都重构为同一个（或高度相似的）分子；
2. **KL 散度先验正则化**：在损失函数中加入后验分布 $q_\phi(\mathbf{z} \mid \mathbf{x})$ 与标准正态先验 $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$ 之间的 Kullback-Leibler（KL）散度惩罚，阻止编码器把不同分子推向无穷远，也不能把方差 $\boldsymbol{\sigma}^2$ 缩成零。

因此，$\beta$-VAE 的单样本总损失函数（即负证据下界 Negative ELBO 的加权形式）由**重构项**与 **KL 正则项**组成：

$$\mathcal{L}_{\beta\text{-VAE}}(\mathbf{x}) = \underbrace{\mathcal{L}_{\text{recon}}(\mathbf{x}, \hat{\mathbf{x}})}_{\text{交叉熵重构误差（记得准）}} + \beta \cdot \underbrace{D_{\text{KL}}\!\left(q_\phi(\mathbf{z} \mid \mathbf{x}) \,\|\, \mathcal{N}(\mathbf{0}, \mathbf{I})\right)}_{\text{潜空间向标准正态靠拢（排得密）}}$$

对于 $d$ 维对角高斯分布，KL 散度有闭合解析解：

$$D_{\text{KL}}\!\left(\mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2)) \,\|\, \mathcal{N}(\mathbf{0}, \mathbf{I})\right) = -\frac{1}{2}\sum_{j=1}^{d} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

**重参数化技巧（Reparameterization Trick）**：直接调用随机采样算子 $\mathbf{z} \sim \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ 是不可导的，反向传播梯度无法穿过随机采样节点。为此，我们先从无参数的固定标准正态分布采样 $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$，再通过确定性的仿射变换构造潜向量：

$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon} = \boldsymbol{\mu} + \exp\!\left(\tfrac{1}{2}\log \boldsymbol{\sigma}^2\right) \odot \boldsymbol{\epsilon}$$

这样随机性完全隔离在外生变量 $\boldsymbol{\epsilon}$ 中，损失函数对 $\boldsymbol{\mu}$ 和 $\log \boldsymbol{\sigma}^2$ 的偏导数便可顺畅回传。

---
## 3. 构建与训练 `MolecularVAE`

我们将每条长度为 `max_len` 的 SELFIES 序列做 One-Hot 编码并展平为维度 `INPUT_DIM = max_len * len(alphabet)` 的向量，留出 3,000 个分子作为独立测试集，其余约 27,000 个分子作为训练集。

编码器与解码器均采用三层感知机（MLP），潜空间维度设为 `LATENT_DIM = 128`。为了避免训练初期解码器尚未成型时就被过强的 KL 惩罚压垮（即发生**后验坍缩 Posterior Collapse**，编码器直接令 $\boldsymbol{\mu}=\mathbf{0}, \boldsymbol{\sigma}=\mathbf{1}$ 放弃传递分子信息），我们在前 13 个 Epoch 采用线性 **KL Warm-up（退火）** 策略，将权重从 $0$ 逐步升至目标值 $\beta = 0.05$。

In [ ]:
indices = torch.tensor([[stoi[t] for t in ts] + [0] * (max_len - len(ts)) for ts in token_lists], dtype=torch.long)
X = F.one_hot(indices, len(alphabet)).float().reshape(len(indices), -1)
print(f"Input matrix shape: {tuple(X.shape)} ({max_len} positions x {len(alphabet)} tokens)")

perm = np.random.RandomState(0).permutation(len(X))
test_idx, train_idx = perm[:3000], perm[3000:]
print(f"Train samples: {len(train_idx):,} | Test samples: {len(test_idx):,}")

In [ ]:
LATENT_DIM = 128
INPUT_DIM = X.shape[1]

class MolecularVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        # 重参数化技巧：z = mu + sigma * eps
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z), mu, logvar

vae = MolecularVAE(INPUT_DIM, LATENT_DIM)
print(f"MolecularVAE total parameters: {sum(p.numel() for p in vae.parameters()):,}")

In [ ]:
def decode_latent(z, model=vae):
    """将一批潜向量 z 解码为 SMILES 字符串列表。"""
    model.eval()
    with torch.no_grad():
        logits = model.decoder(z)
    token_ids = logits.reshape(len(z), max_len, len(alphabet)).argmax(dim=-1)
    out = []
    for row in token_ids.tolist():
        s = "".join(alphabet[i] for i in row if i != 0)  # 跳过 [nop] 填充符
        out.append(sf.decoder(s))
    return out

def canonical(smiles):
    """若能被 RDKit 合法解析则返回 Canonical SMILES，否则返回 None。"""
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None and mol.GetNumAtoms() > 0 else None

为节省课堂等待时间，下方单元默认直接加载已在相同训练集上完成 40 轮训练的权重 `data/cep_vae.pt`（含完整损失历史）；如果你想亲手观察从头训练的过程，只需将 `TRAIN_FROM_SCRATCH` 设为 `True`（CPU 上约需 3 分钟）。

In [ ]:
BETA = 0.05
TRAIN_FROM_SCRATCH = False
vae_ckpt_path = os.path.join(DATA_DIR, "cep_vae.pt")

if os.path.exists(vae_ckpt_path) and not TRAIN_FROM_SCRATCH:
    ckpt_vae = torch.load(vae_ckpt_path, map_location="cpu")
    vae.load_state_dict(ckpt_vae["model_state_dict"])
    history = ckpt_vae["history"]
    vae.eval()
    print(f"Loaded pretrained MolecularVAE from {vae_ckpt_path}")
else:
    optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
    history = []
    vae.train()
    for epoch in range(40):
        order = np.random.permutation(train_idx)
        rec_total = kl_total = 0.0
        beta = BETA * min(1.0, (epoch + 1) / 13.0)
        for i in range(0, len(order), 256):
            xb = X[order[i : i + 256]]
            out, mu, logvar = vae(xb)
            rec = F.cross_entropy(
                out.reshape(-1, len(alphabet)),
                xb.reshape(-1, len(alphabet)).argmax(-1),
                reduction="sum",
            ) / len(xb)
            kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(xb)
            loss = rec + beta * kl
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            rec_total += rec.item()
            kl_total += kl.item()
        n_batches = len(order) / 256.0
        history.append({"epoch": epoch, "reconstruction": rec_total / n_batches, "KL": kl_total / n_batches})
        if epoch % 5 == 4 or epoch == 0:
            print(f"Epoch {epoch:02d} | Recon = {rec_total / n_batches:6.2f} | KL = {kl_total / n_batches:6.2f}")
    vae.eval()

In [ ]:
hist = pd.DataFrame(history).set_index("epoch")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
hist["reconstruction"].plot(ax=axes[0], lw=2, title="Cross-Entropy Reconstruction Loss")
hist["KL"].plot(ax=axes[1], lw=2, color="tab:red", title="KL Divergence D_KL(q(z|x) || N(0,I))")
for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

观察上方的训练曲线可以看到两项损失的博弈过程：前几个 Epoch 由于 KL Warm-up 权重很小，编码器全力降低重构误差，将不同分子尽量分开，因此 KL 散度快速上升；随着 $\beta$ 逐步增大并稳定在 `0.05`，KL 正则项开始把分散的后验分布往原点附近拉拢，同时重构误差继续稳步收敛。

### 3.1 重构保真度检验：模型记住了分子的哪些特征？

我们取测试集前 1,000 个未见过的分子，将其编码为后验均值 $\boldsymbol{\mu}_{\text{test}}$ 后直接解码还原，统计其有效率与 Canonical SMILES 完全一致率（Exact Reconstruction Rate）。

In [ ]:
with torch.no_grad():
    mu_test, logvar_test = vae.encode(X[test_idx])

reconstructed = decode_latent(mu_test[:1000])
originals = [canonical(s) for s in df["smiles"].values[test_idx[:1000]]]
recon_canonical = [canonical(s) for s in reconstructed]

exact = np.mean([a is not None and a == b for a, b in zip(recon_canonical, originals)])
valid = np.mean([a is not None for a in recon_canonical])
print(f"Test Reconstruction Validity   : {valid:.3f}")
print(f"Exact Canonical Match Rate     : {exact:.3f}")

pd.DataFrame([
    {"Original SMILES": o, "Reconstructed SMILES": r, "Exact Match": o == r}
    for o, r in zip(originals[:6], recon_canonical[:6])
])

In [ ]:
pairs = [
    (Chem.MolFromSmiles(o), Chem.MolFromSmiles(r))
    for o, r in zip(originals, recon_canonical)
    if o and r and o != r
][:4]
Draw.MolsToGridImage(
    [m for pair in pairs for m in pair],
    molsPerRow=2,
    subImgSize=(280, 200),
    legends=[lbl for _ in range(len(pairs)) for lbl in ("Original", "Reconstructed")],
)

得益于 SELFIES 的形式语法保障，重构分子的有效率接近 100%；而完全一字不差还原出原分子的比例约为 18%。对照上方画出的分子对可以看出：模型对于未完全还原的样本并非胡乱猜测，而是**准确还原了稠环共轭主骨架的尺寸与连接方式，仅在局部杂原子类型（例如把噻吩环 `s` 替换为呋喃环 `o` 或吡咯环 `[nH]`）或环位置上出现了细微偏差**。

造成这种局部替换的原因在于我们使用了最简单的单步 MLP 解码器——所有位置的 token 是在同一时刻独立取 `argmax` 输出的，位置 $t$ 并不知道位置 $t-1$ 刚刚选了什么符号。工业界更高精度的分子 VAE（如 Gómez-Bombarelli et al. 或 JT-VAE）通常采用自回归 GRU/Transformer 解码器或基于分子图树分解的解码器，以换取更高的逐原子还原率。

### 练习 09a-1：验证重构精度与潜空间规整度的权衡（$\beta = 0.05$ vs. $\beta = 0.5$）

在 $\beta$-VAE 的损失函数中，超参数 $\beta$ 直接控制着“解码器还原得准”与“潜空间分布接近标准正态先验 $\mathcal{N}(\mathbf{0}, \mathbf{I})$”之间的拉锯。

请在下方编写代码，在训练集的 5,000 个子样本上分别以 $\beta = 0.05$ 和 $\beta = 0.5$（增大 10 倍）各训练一个 `MolecularVAE` 15 个 Epoch，对比两者的：
1. 训练结束时的重构误差（`Reconstruction Loss`）与 KL 散度（`KL Divergence`）；
2. 测试集前 500 个样本的完全还原率（`Exact Match Rate`）；
3. 测试集编码均值 $\boldsymbol{\mu}_{\text{test}}$ 每一维的标准差均值 `mu_test.std(dim=0).mean()`（越接近 `1.0` 说明全局分布越贴合先验，越接近 `0.0` 说明发生了后验坍缩）。

In [ ]:
# 练习 09a-1：在此对比 beta = 0.05 与 beta = 0.5 的重构—规整权衡
def compare_beta_vae(beta_values=(0.05, 0.5), n_epochs=15):
    # TODO:
    # 1. 对每个 beta 初始化新的 MolecularVAE(INPUT_DIM, LATENT_DIM)
    # 2. 在 train_idx[:5000] 上训练 n_epochs 轮
    # 3. 统计测试集 mu_test 的重构率与 mu_test.std(0).mean()
    pass


---
## 4. 潜空间高维几何：为什么欧氏直线插值会发生“范数塌陷”？

### 4.1 两个分子之间的潜空间插值

获得了连续潜向量表示后，最直观的操作是在两个测试集分子 $\mathbf{z}_A = \boldsymbol{\mu}_A$ 与 $\mathbf{z}_B = \boldsymbol{\mu}_B$ 之间做线性插值（Linear Interpolation, Lerp）：

$$\mathbf{z}_{\text{lerp}}(t) = (1 - t)\,\mathbf{z}_A + t\,\mathbf{z}_B, \qquad t \in [0, 1]$$

我们取出两个测试集分子，沿直线均匀采样 8 个点并解码画图，观察中间过渡分子的形态。

In [ ]:
idx_a, idx_b = 0, 7
z_a, z_b = mu_test[idx_a], mu_test[idx_b]

steps = torch.linspace(0, 1, 8).unsqueeze(1)
z_path = (1.0 - steps) * z_a + steps * z_b
path_smiles = decode_latent(z_path)

path_mols, legends = [], []
for t_val, s in zip(steps.flatten().tolist(), path_smiles):
    c = canonical(s)
    if c:
        path_mols.append(Chem.MolFromSmiles(c))
        legends.append(f"t = {t_val:.2f}")
Draw.MolsToGridImage(path_mols, molsPerRow=4, subImgSize=(250, 190), legends=legends)

In [ ]:
# 检查直线插值路径上每一步潜向量的欧氏范数 ||z(t)||_2
norms = z_path.norm(dim=1)
for t_val, n_val in zip(steps.flatten().tolist(), norms.tolist()):
    print(f"t = {t_val:.2f} | ||z(t)||_2 = {n_val:.2f}")
print(f"\nMedian ||mu||_2 of encoded CEP test molecules: {mu_test.norm(dim=1).median():.2f}")

观察上方的插值图与范数打印结果，一个反直觉的现象浮现出来：两端（$t=0$ 与 $t=1$）都是规整的稠环芳香体系，但中间过渡点（$t \in [0.29, 0.71]$）却突然崩解成了开链或怪异的大环结构！

为什么直线插值在中间会失真？答案藏在**高维欧氏空间的几何性质**中：
在 $d = 128$ 维空间里，两个独立编码的向量 $\mathbf{z}_A$ 与 $\mathbf{z}_B$ 几乎总是近似正交的（夹角接近 $90^\circ$）。假设两者的模长均为 $\|\mathbf{z}_A\|_2 \approx \|\mathbf{z}_B\|_2 = r$ 且相互正交，那么它们直线中点 $\mathbf{z}_{\text{lerp}}(0.5) = \frac{1}{2}(\mathbf{z}_A + \mathbf{z}_B)$ 的模长为：

$$\left\|\frac{\mathbf{z}_A + \mathbf{z}_B}{2}\right\|_2 = \frac{\sqrt{\|\mathbf{z}_A\|_2^2 + \|\mathbf{z}_B\|_2^2}}{2} = \frac{r}{\sqrt{2}} \approx 0.707\,r$$

正如上方打印所示，两端真实分子的模长在 `8.5` 左右，而直线中点的模长塌陷到了 `6.1` 左右！高维高斯型分布的样本绝大部分集中在一层薄薄的超球壳（Hyperspherical Shell）上，球壳内部靠近原点的区域实际上是没有训练样本分布的“空腔”。直线插值相当于直接切过超球体内部的空腔，解码器在从未见过的空腔区域自然无法输出合理的共轭分子。

要让插值路径始终贴合高密度球壳表面，标准做法是改用**球面线性插值（Spherical Linear Interpolation, Slerp）**，按夹角 $\omega = \arccos(\hat{\mathbf{z}}_A \cdot \hat{\mathbf{z}}_B)$ 在大圆弧上匀速旋转。

### 4.2 潜空间的宏观组织结构：按结构聚类还是按性能聚类？

我们复用 `03a` 的主成分分析（PCA），将测试集 3,000 个分子的 128 维潜向量 $\boldsymbol{\mu}_{\text{test}}$ 投影到二维平面，分别按光电转换效率（`PCE`）和分子尺寸（重原子数 `Heavy Atoms`）着色。

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(mu_test.numpy())
pce_test = df["PCE"].values[test_idx]
heavy_test = np.array([Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in df["smiles"].values[test_idx]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sc0 = axes[0].scatter(coords[:, 0], coords[:, 1], c=pce_test, cmap="viridis", s=5, alpha=0.8)
plt.colorbar(sc0, ax=axes[0], label="DFT PCE (%)")
axes[0].set_title("VAE Latent Space (PCA) Colored by PCE")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

sc1 = axes[1].scatter(coords[:, 0], coords[:, 1], c=heavy_test, cmap="plasma", s=5, alpha=0.8)
plt.colorbar(sc1, ax=axes[1], label="Heavy Atom Count")
axes[1].set_title("VAE Latent Space (PCA) Colored by Molecular Size")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.show()
print("Variance explained by PC1 & PC2:", pca.explained_variance_ratio_.round(3))

可以看到：右图中按重原子数着色的潜空间呈现出极其清晰的梯度分层（小分子与大分子各居一侧），而左图中按 `PCE` 着色的分布则混杂在一起。原因在于：**无监督 VAE 在训练阶段从未见过 `PCE` 标签**，它的目标函数纯粹是重建 SELFIES 序列，因此潜空间完全按照“拓扑结构与序列相似性”组织；若想让潜空间沿某条轴线直接对齐目标性质，需要在训练 VAE 时并联一个从 $\mathbf{z}$ 预测性质的回归头进行联合训练（Joint Property-VAE）。

### 练习 09a-2：球面线性插值（Slerp）修正范数塌陷与潜空间梯度外推实验

本练习包含两个紧密相连的小实验：

**(a) 实现球面线性插值 `slerp(z_a, z_b, t)`**：
给定两个潜向量 $\mathbf{z}_A, \mathbf{z}_B$，先将它们归一化为单位向量 $\hat{\mathbf{z}}_A = \mathbf{z}_A / \|\mathbf{z}_A\|_2$、$\hat{\mathbf{z}}_B = \mathbf{z}_B / \|\mathbf{z}_B\|_2$，计算夹角 $\omega = \arccos(\text{clamp}(\hat{\mathbf{z}}_A \cdot \hat{\mathbf{z}}_B, -1, 1))$，再结合线性过渡的半径 $r(t) = (1-t)\|\mathbf{z}_A\|_2 + t\|\mathbf{z}_B\|_2$ 计算：
$$\mathbf{z}_{\text{slerp}}(t) = r(t) \cdot \frac{\sin((1-t)\omega)\,\hat{\mathbf{z}}_A + \sin(t\omega)\,\hat{\mathbf{z}}_B}{\sin(\omega)}$$
打印 $t \in [0, 1]$ 上 8 个插值点的范数 $\|\mathbf{z}_{\text{slerp}}(t)\|_2$，验证范数塌陷已被彻底消除，并统计解码分子的平均芳香环个数相比直线插值是否有改善。

**(b) 潜空间线性回归梯度外推（预演 `09b` 的代理模型欺骗）**：
用 `mu_test`（128 维）和 `pce_test` 训练一个岭回归器 `Ridge(alpha=1.0)`。接着从测试集中 PCE 最高的那个分子潜向量 $\mathbf{z}_0$ 出发，沿着岭回归权重方向 $\hat{\mathbf{w}} = \mathbf{w} / \|\mathbf{w}\|_2$ 走出步长 $\alpha \in \{0, 2, 5, 10, 20\}$：
$$\mathbf{z}(\alpha) = \mathbf{z}_0 + \alpha\,\hat{\mathbf{w}}$$
观察每一步岭回归预测的 PCE 如何飙升，而解码出的实际分子结构（如芳香环数、重原子数）发生了什么畸变？

In [ ]:
# 练习 09a-2：在此实现 (a) slerp 球面插值 与 (b) 潜空间 Ridge 梯度上升外推
def slerp(a, b, t):
    # TODO: 实现球面线性插值
    pass


---
## 5. 从先验分布采样 vs. 局部邻域扰动：为什么“100% 合法”不等于“合理材料”？

### 5.1 先验采样的假象：满分三项指标背后的分布崩塌

按照 VAE 的理论设计，训练完成后我们可以丢掉编码器，直接从标准正态先验 $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ 中随机抽取潜向量并解码生成新分子。我们抽取 1,000 个先验样本，先看生成模型论文中最常汇报的三个标准字符串指标：**有效性（Validity）**、**唯一性（Uniqueness）**与**新颖性（Novelty）**。

In [ ]:
torch.manual_seed(0)
z_prior = torch.randn(1000, LATENT_DIM)
prior_smiles = decode_latent(z_prior)
prior_canonical = [canonical(s) for s in prior_smiles]
prior_valid = [c for c in prior_canonical if c]

canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}
print(f"Prior Sampling Validity   : {len(prior_valid) / 1000:.3f}")
print(f"Prior Sampling Uniqueness : {len(set(prior_valid)) / max(len(prior_valid), 1):.3f}")
print(f"Prior Sampling Novelty    : {len(set(prior_valid) - canonical_train) / max(len(set(prior_valid)), 1):.3f}")

如果只看这三行数字——有效率 >90%、唯一性与新颖性接近 100%——这似乎是一个无可挑剔的分子生成模型。但别忘了第 1 节的结论：SELFIES 哪怕面对纯随机乱码也能保证极高的语法有效率！

让我们把先验采样解码出的前 12 个分子画出来，并与 CEP 训练数据对比三个核心化学结构描述符：**重原子数（Heavy Atoms）**、**芳香环数（Aromatic Rings）**与 **$sp^3$ 杂化碳比例（FractionCSP3）**。

In [ ]:
Draw.MolsToGridImage([Chem.MolFromSmiles(c) for c in prior_valid[:12]], molsPerRow=4, subImgSize=(260, 200))

In [ ]:
train_sample = [Chem.MolFromSmiles(s) for s in df["smiles"].sample(2000, random_state=0)]
prior_mols = [Chem.MolFromSmiles(c) for c in prior_valid]

comparison = pd.DataFrame({
    "CEP Training Data": [
        np.mean([m.GetNumHeavyAtoms() for m in train_sample]),
        np.mean([Descriptors.NumAromaticRings(m) for m in train_sample]),
        np.mean([Descriptors.FractionCSP3(m) for m in train_sample]),
    ],
    "VAE Prior Samples z ~ N(0,I)": [
        np.mean([m.GetNumHeavyAtoms() for m in prior_mols]),
        np.mean([Descriptors.NumAromaticRings(m) for m in prior_mols]),
        np.mean([Descriptors.FractionCSP3(m) for m in prior_mols]),
    ],
}, index=["Heavy Atoms", "Aromatic Rings", "Fraction sp3 C"]).round(2)
comparison

分子网格图与定量对比表揭示了惊人的反差：
- CEP 训练数据是典型的有机光伏稠环共轭体系，平均每个分子含有 **5.73 个芳香环**，$sp^3$ 饱和碳比例仅为 **0.02**；
- 从 VAE 先验 $\mathcal{N}(\mathbf{0}, \mathbf{I})$ 采样出的分子，平均芳香环数暴跌至 **不到 0.5 个**，到处是破碎的脂肪链、三元/四元张力环与孤立杂原子！虽然它们的化合价在 SELFIES 状态机强制修正下没有报错，但作为有机光伏材料毫无物理意义。

### 5.2 诊断根源：先验空洞（Prior Hole / Aggregated Posterior Mismatch）

为什么从标准正态先验 $\mathcal{N}(\mathbf{0}, \mathbf{I})$ 采样会彻底偏离训练数据？这是变分自编码器中最著名的问题——**先验空洞（Prior Hole）**：

KL 正则项只是软性地拉近每个样本的后验分布与 $\mathcal{N}(\mathbf{0}, \mathbf{I})$，在有限权重 $\beta = 0.05$ 下，所有训练样本后验分布叠加而成的**聚合后验（Aggregated Posterior）** $q_\phi(\mathbf{z}) = \frac{1}{N}\sum_{i=1}^N q_\phi(\mathbf{z} \mid \mathbf{x}_i)$ 并未铺满整个 $\mathcal{N}(\mathbf{0}, \mathbf{I})$。我们在高维空间中直接测量两者的半径分布即可一目了然：

In [ ]:
print(f"Mean per-dimension std of encoded mu_test : {mu_test.std(0).mean():.3f}  (Standard Normal prior expects 1.000)")
print(f"Median norm ||mu_test||_2 of real molecules: {mu_test.norm(dim=1).median():.2f}")
print(f"Median norm ||z_prior||_2 of N(0,I) samples: {torch.randn(3000, LATENT_DIM).norm(dim=1).median():.2f}  ( approx sqrt(128) = {math.sqrt(128):.2f} )")

在 128 维空间中，标准正态分布 $\mathcal{N}(\mathbf{0}, \mathbf{I})$ 的样本几乎全部集中在半径 $\sqrt{128} \approx 11.3$ 的外层球壳上；而真实 CEP 分子的编码均值 $\boldsymbol{\mu}_{\text{test}}$ 每一维标准差仅为 `0.55`，实际居住在半径 `8.5` 左右的内层球壳上！
换言之：
- 直线插值的中点掉进了半径 $\approx 6.1$ 的**球壳内部空腔**；
- 标准正态先验采样全部落在了半径 $\approx 11.3$ 的**球壳外部荒原**；
- 真正的有机光伏分子只居住在半径 $\approx 8.5$ 的那层薄壳上！

### 5.3 实用解决方案：以已知种子分子为中心的局部邻域采样

既然真实分子只居住在特定流形薄壳上，在工程实践中用 VAE 生成候选分子时，我们不应盲目从全局先验 $\mathcal{N}(\mathbf{0}, \mathbf{I})$ 抽样，而应**以已知真实分子的潜坐标 $\boldsymbol{\mu}_i$ 为中心，叠加适度的高斯扰动 $\sigma$ 进行局部探索**：

$$\mathbf{z} = \boldsymbol{\mu}_i + \sigma \cdot \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

In [ ]:
rows_local = []
base_mu = mu_test[:300]
originals_300 = [canonical(s) for s in df["smiles"].values[test_idx[:300]]]

for sigma in [0.0, 0.3, 1.0, 3.0]:
    z_pert = base_mu + sigma * torch.randn_like(base_mu)
    out = [canonical(s) for s in decode_latent(z_pert)]
    ok = [c for c in out if c]
    mols_ok = [Chem.MolFromSmiles(c) for c in ok]
    rows_local.append({
        "Noise Std (sigma)": sigma,
        "Validity": round(len(ok) / len(base_mu), 3),
        "Identical to Seed": round(float(np.mean([a == b for a, b in zip(out, originals_300)])), 3),
        "Mean Aromatic Rings": round(float(np.mean([Descriptors.NumAromaticRings(m) for m in mols_ok])), 2),
        "Mean Heavy Atoms": round(float(np.mean([m.GetNumHeavyAtoms() for m in mols_ok])), 1),
    })

pd.DataFrame(rows_local).set_index("Noise Std (sigma)")

当局部扰动幅度取 $\sigma = 0.3$ 时，超过 90% 的解码分子是不同于种子分子的**全新分子**，同时平均芳香环数（约 4 个）与重原子数（约 26 个）依然保持了稠环共轭有机半导体的合理结构特征；而当 $\sigma$ 增大到 `3.0` 时，采样点再次飞出有效流形薄壳，退化为与先验采样相同的非共轭碎片。

### 练习 09a-3：“完美背诵模型” vs. VAE 先验采样的生成指标盲区诊断

为了深刻体会为什么评估生成模型时绝不能只看 `Validity / Uniqueness / Novelty`，请设计一个对比实验：
1. 从训练集 `df["smiles"].iloc[train_idx]` 中随机抽出 1,000 个真实分子，将其视为一个“只会照抄训练集的假生成模型（Memorization Baseline）”；
2. 计算这个“照抄模型”与 `z_prior`（VAE 先验采样）在六个维度上的得分并汇总为一张 DataFrame：
   - 字符串三指标：`Validity`、`Uniqueness`、`Novelty`（相对于 `canonical_train`）；
   - 结构分布三指标：`Mean Heavy Atoms`、`Mean Aromatic Rings`、`Mean FractionCSP3`。
3. 思考：为什么在 `09c` 即将介绍的**去噪扩散概率模型（DDPM）**中，先验空洞问题能够从根源上得到解决？

In [ ]:
# 练习 09a-3：在此对比“训练集照抄基线”与 VAE 先验采样的六项指标
# TODO: 构造包含 Validity, Uniqueness, Novelty, Heavy Atoms, Aromatic Rings, FractionCSP3 的对比表


---
## 6. 本节小结：自回归语言模型 vs. 变分自编码器

| 维度 | SMILES 自回归语言模型（`08a`/`08b`） | SELFIES 变分自编码器（`09a`） |
| :--- | :--- | :--- |
| **生成方式与语法保障** | 逐 token 条件自回归；靠学习长程依赖维持约 95% 有效率 | 潜向量并行解码；借助 SELFIES 形式语法强制保证 ~100% 化合价合法 |
| **分布保真度（先验采样）** | **极高**（描述符与构筑单元频率与训练集高度重合） | **易受先验空洞影响**（全局先验采样偏离训练流形，需在种子分子邻域 $\sigma \approx 0.3$ 采样） |
| **显式连续潜坐标 $\mathbf{z}$** | 无（隐状态随序列长度动态变化） | **有**（固定维度欧氏向量，支持球面插值 Slerp 与局部邻域搜索） |
| **核心评估铁律** | `Validity / Uniqueness / Novelty` 只是底线门槛，**必须结合结构描述符分布、构筑单元频率与可视化分子图**检验化学合理性 |

无论是 `08a` 的语言模型还是本节的 `09a` 变分自编码器，它们目前学习的都只是“如何生成像训练数据的分子”。如果我们不满足于复制现有分布，而是希望定向生成**光电转换效率（PCE）显著高于现有数据库平均水平**的全新分子，该如何把生成模型与性质预测模型结合起来？请进入下一节 **`09b · 目标导向的分子逆向设计`**。